## TIFON Databases. Read and load example databases

In [ ]:
try:
    import pyLOM
    print('Environment with pylom instaled')
except ImportError:
    import sys
    sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

def read_db(datafolder, case_idx, make_data_dict:bool = True):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    if make_data_dict:
        db.extract_inputs(
            id_groups = (3,),
            cases_idx = case_idx,
            vtu_type='surface',
            verbose=False
            )

        for stage in [0, 1]:
            
            db.extract_outputs(
                id_groups=(3,),
                stage=stage, cases_idx = case_idx,
                var_name_excluded = [
                    'BoundaryValues_CoefSkinFrictionX',
                    'BoundaryValues_CoefSkinFrictionY',
                    'BoundaryValues_CoefSkinFrictionZ'
                    ],
                vtu_type='surface',
                )
    
    return db

In [ ]:
db_1 = read_db('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/', case_idx='all', make_data_dict=False)

### Residuals

In [ ]:
db_1.plot_state()

In [ ]:
db_1.residuals.plot_all_final_residuals(
    mode='scaled',
    stage='all',
    only_finished=True,
    activate_idx = True,
)

In [ ]:
for c in [18, 24, 36, 39, 65, 70, 76]:
    db_1.residuals.plot_residuals_from_case(
        case_idx = c,
        stage='all',
        mode='scaled',
    )

In [ ]:
db_1.residuals.plot_state_calculation(num_stages=1, txt_from_end=1)

### Calculate df_post from get_df_metrics()

In [ ]:
for c in range(95,100):
    db_1.residuals.plot_residuals_from_case(
        case_idx = c,
        stage = [0, 1],
        mode = 'scaled'
    )

In [ ]:
db_1.residuals.plot_all_final_residuals(mode='scaled', stage=[0,], lim_converged=1e-4, only_finished=True, print_non_converged=False, activate_idx=True)

In [ ]:
db_1.residuals.plot_all_final_residuals(mode='scaled', stage=[0, 1], lim_converged=1e-5, only_finished=True, print_non_converged=False, activate_idx=True)

In [ ]:
all = list(range(80))
to_remove = [0, 3, 4, 6, 7, 9, 13, 15, 16, 19, 25, 28, 31, 39, 42, 45, 50, 54, 60, 63, 69, 72, 75]
print(1-(len(to_remove)/len(all)))

In [ ]:
res_mean, res_std = db_1.residuals.integrals_convergence_criteria(
    iterations_back = 1000,
    only_finished = True,
    only_converged = False, 
    mode = '2D',
    plot=True,
    residual_threshold = 1e-3,
    verbose=True
)

In [ ]:
df_finals = db_1.residuals.get_all_final_residuals(verbose=False, stage=[0,], only_finished=False, load_in_metadata=False)
display(df_finals)

In [ ]:
db_1.metadata

In [ ]:
df_post = db_1.residuals.get_df_metrics(var_metrics = ['CoefLift', 'CoefDrag', 'CoefMomentY'],
        iter_var = 1000,
        save =  True
)

### Analizar df_post

In [ ]:
import pandas as pd
df_post = pd.read_csv(
    '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_#280/metadata/df_post.csv',
    sep=',',
    index_col=0   
)
# for stage in [0, 1]:
#     col=f'coeflift_mean_stage{stage}'
#     arr = df_post.loc[df_post['dataset'] == 'dataset_3', col] * 0.1
#     df_post.loc[df_post['dataset'] == 'dataset_3', col] = arr
    
#     col=f'coefdrag_mean_stage{stage}'
#     arr = df_post.loc[df_post['dataset'] == 'dataset_3', col] * 0.1
#     df_post.loc[df_post['dataset'] == 'dataset_3', col] = arr
    
# df_post.to_csv(
#     '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_#280/metadata/df_post.csv',
#     sep=',',
#     index_label = 'index'  
# )
# df_post['size'] = df_post['densityresidual_scaled_stage1'].apply(lambda x: -np.log10(x))
# df_post['size'] = 10*(df_post['size'] - df_post['size'].min()) / (df_post['size'].max() - df_post['size'].min())
# display(df_post[['aoa', 'mach', 'size', 'dataset']])
# print(df_post['size'].min(), df_post['size'].max())

In [ ]:
import plotly.express as px
import numpy as np
from typing import Union

def plot_df_post(df_post, col1:str, col2:str, col3:str = None, line_xy:bool = True, save_path:Union['str', bool] = False):
    
    df_post['size'] = df_post['densityresidual_scaled_stage1'].apply(lambda x: 1/-np.log10(x))
    
    df_post['size'] = 10*(df_post['size'] - df_post['size'].min()) / (df_post['size'].max() - df_post['size'].min())
    dicc_batch = {
        'dataset_0': 'batch_0',
        'dataset_1': 'batch_0',
        'dataset_2': 'batch_1',
        'dataset_3': 'batch_2',
    }
    
    df_post['batch'] = df_post['dataset'].apply(lambda x: dicc_batch[x])
    if col3 is not None:
        fig = px.scatter(
            df_post,
            x=col1, y=col2,
            color=col3,
            symbol="dataset",
            symbol_sequence=["circle", "diamond", "triangle-up", "square"],
            hover_data=df_post.columns,
            #tamaño del marker inversamente proporcional al valor de la columna densityresidual_stage0 
            size=df_post['size']
        )
    else:
        fig = px.scatter(
            df_post, 
            x=col1, y=col2, 
            symbol="dataset",
            symbol_sequence=["circle", "diamond", "triangle-up", "square"],
            hover_data=df_post.columns
            )
    
    if line_xy:
        min_val = min(df_post[col1].min(), df_post[col2].min())
        max_val = max(df_post[col1].max(), df_post[col2].max())
        fig.add_shape(type='line', x0=min_val, y0=min_val, x1=max_val, y1=max_val,
                      line=dict(color='Red', dash='dash'))
    
    fig.update_layout(
        title=f'{col1} vs {col2}',
        xaxis_title=col1,
        yaxis_title=col2,
        height=500, #
        width=800
        )
    # poner la leyenda a la izquierda
    fig.update_layout(legend=dict(x=0, y=1))
    #guardar figura como png si save_path es un string
    if save_path:
        fig.write_image(save_path)
        
    else:
        fig.show()

In [ ]:
plot_df_post(df_post, col1='coeflift_mean_stage0', col2='coeflift_mean_stage1', col3='batch', line_xy=True, save_path='/home/m.jaraiz/repos/tmp/cl0_vs_cl1.png')
plot_df_post(df_post, col1='coefdrag_mean_stage0', col2='coefdrag_mean_stage1', col3='batch', line_xy=True, save_path='/home/m.jaraiz/repos/tmp/cd0_vs_cd1.png')
for stage in [0, 1]:

    plot_df_post(df_post, col1='aoa', col2=f'coeflift_mean_stage{stage}', col3='batch', line_xy=False, save_path=f'/home/m.jaraiz/repos/tmp/aoa_vs_cl{stage}.png')
    plot_df_post(df_post, col1=f'coeflift_mean_stage{stage}', col2=f'coefdrag_mean_stage{stage}', col3='batch', line_xy=False, save_path=f'/home/m.jaraiz/repos/tmp/cl{stage}_vs_cd{stage}.png')
    plot_df_post(df_post, col1='mach', col2=f'coefdrag_mean_stage{stage}', col3='batch', line_xy=False, save_path=f'/home/m.jaraiz/repos/tmp/mach_vs_cd{stage}.png')

In [ ]:

fig = px.scatter(
    df_post,
    x='CoefDrag_mean_stage0',
    y='CoefDrag_mean_stage1',
    color='mach',
    hover_data=['case_idx', 'mach']
)

fig.show()
fig = px.scatter(
    df_post,
    x='CoefLift_mean_stage0',
    y='CoefLift_mean_stage1',
    color='mach',
    hover_data=['case_idx', 'mach']
)
fig.show()
fig = px.scatter(
    df_post,
    x="aoa",
    y="CoefLift_mean_stage1",
    color="mach",
    # title="CL vs AoA"
)
fig.show()
fig = px.scatter(
    df_post,
    x="CoefDrag_mean_stage1",
    y="CoefLift_mean_stage1",
    color="aoa"
)
fig.show()
fig = px.scatter(
    df_post,
    x="CoefDrag_mean_stage1",
    y="CoefLift_mean_stage1",
    color="mach"
)
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# -------------------------
# Renderer para VSCode
# -------------------------
pio.renderers.default = "vscode"

# -------------------------
# Crear subplots
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Stage0 vs Stage1 (CL)",
        "Polar CL vs AoA",
        "Drag Polar (CL vs CD)"
    )
)

# -------------------------
# 1. CL Stage0 vs Stage1
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_post['CoefLift_mean_stage0'],
        y=df_post['CoefLift_mean_stage1'],
        mode='markers',
        marker=dict(
            color=df_post['aoa'],
            colorscale='Viridis',
            colorbar=dict(title="AoA"),
            showscale=True
        ),
        text=df_post['case_idx'],
        customdata=df_post[['mach']],
        hovertemplate=
        "CL0=%{x}<br>CL1=%{y}<br>AoA=%{marker.color}<br>Mach=%{customdata[0]}<br>Case=%{text}",
        name="CL stage comparison"
    ),
    row=1, col=1
)

# -------------------------
# 2. Polar CL vs AoA
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_post["aoa"],
        y=df_post["CoefLift_mean_stage1"],
        mode="markers",
        marker=dict(
            color=df_post["mach"],
            colorscale="Viridis",
            showscale=False
        ),
        text=df_post["case_idx"],
        hovertemplate=
        "AoA=%{x}<br>CL=%{y}<br>Mach=%{marker.color}<br>Case=%{text}",
        name="Polar"
    ),
    row=1, col=2
)

# -------------------------
# 3. Drag polar
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_post["CoefDrag_mean_stage1"],
        y=df_post["CoefLift_mean_stage1"],
        mode="markers",
        marker=dict(
            color=df_post["aoa"],
            colorscale="Viridis",
            showscale=False
        ),
        text=df_post["case_idx"],
        customdata=df_post[['mach']],
        hovertemplate=
        "CD=%{x}<br>CL=%{y}<br>AoA=%{marker.color}<br>Mach=%{customdata[0]}<br>Case=%{text}",
        name="Drag polar"
    ),
    row=1, col=3
)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=500,
    width=1400,
    title="CFD Interactive Analysis Dashboard",
)

# -------------------------
# Interactividad extra (selector dataset)
# -------------------------
datasets = df_post['dataset'].unique()

buttons = []
for s in datasets:
    mask = df_post['stage'] == s

    visible = []
    for _ in range(3):  # 3 traces
        visible.append(True)

    buttons.append(dict(
        label=s,
        method="update",
        args=[{
            "x": [
                df_post.loc[mask, 'CoefLift_mean_stage0'],
                df_post.loc[mask, 'aoa'],
                df_post.loc[mask, 'CoefDrag_mean_stage1']
            ],
            "y": [
                df_post.loc[mask, 'CoefLift_mean_stage1'],
                df_post.loc[mask, 'CoefLift_mean_stage1'],
                df_post.loc[mask, 'CoefLift_mean_stage1']
            ],
        }]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            y=1.2,
            showactive=True
        )
    ]
)

fig.show()